In [ ]:
import time
from pathlib import Path

import geopandas as gpd
import rasterio
import openeo
import pandas as pd
import numpy as np
import os

from scipy.ndimage import median_filter
from rasterio.mask import mask

from datetime import datetime

In [ ]:
# Remove connection to the backend and remove the refresh token 
#from openeo.rest.auth.config import RefreshTokenStore
#RefreshTokenStore().remove()

In [ ]:
url = "https://openeo.dataspace.copernicus.eu"
connection = openeo.connect(url).authenticate_oidc()

## 1) Input setting and preparation

In [ ]:
year = 2023

start_date, end_date = ["2023-07-01", "2023-09-30"]

from pathlib import Path
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/" + str(year))
output_folder.mkdir(exist_ok=True)

results_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S1/" + str(year))
results_folder.mkdir(exist_ok=True)

In [ ]:
bursts_filename = '/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/S1_coh/s1_bursts_glacierSouthTyrol.geojson' # Extracted from the burst id map (https://sar-mpc.eu/sar-mpc-products/)
bursts = gpd.read_file(bursts_filename)
bursts.boundary.plot()

## 2) Multi-temporal mosaic of coherence

#### ONE PROCESS GRAPH 

In [ ]:
ascending_orbits = [15, 44, 117]
descending_orbits = [95, 168]
THRESHOLD = 0.4

def burst_cube(b):
    cube = connection.datacube_from_process(
    "sentinel1_sar_coherence",
    namespace="https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/algorithm_catalog/eurac/sentinel1_sar_coherence/openeo_udp/sentinel1_sar_coherence.json",
    temporal_extent=[start_date, end_date],
    temporal_baseline=12,
    burst_id=int(b["burst_id"]),
    sub_swath=b["subswath_name"],
    coherence_window_az=7,
    coherence_window_rg=22,
    polarization="VV",
        )
    return (
    cube
            .reduce_dimension(dimension="t", reducer="max")
            .resample_spatial(projection=32632, resolution=10, method="bilinear")
        )

def merge_all(cubes, resolver="max"):
    merged = None
    for c in cubes:
        merged = c if merged is None else merged.merge_cubes(c, overlap_resolver=resolver)
    return merged

def direction_mosaic(orbits):
    """One mosaic from all bursts of all given relative orbits."""
    selected = bursts[bursts["relative_orbit_number"].isin(orbits)]
    return merge_all([burst_cube(b) for _, b in selected.iterrows()])

def apply_threshold(cube, thr=THRESHOLD):
    # 1 = valid pixel with coherence < thr, 0 = everything else (incl. nodata)
    return cube.apply(lambda x: if_(is_nodata(x), 0, if_(x < thr, 1, 0)))

job_list = []
for name, orbits in [("ascending", ascending_orbits), ("descending", descending_orbits)]:
    mosaic = direction_mosaic(orbits)
    result = apply_threshold(mosaic).save_result(format="GTiff")

    job = result.create_job(title=f"threshold_{name}_{THRESHOLD}_{year}")
    job.start()
    job_list.append(job)
    print(f"Started {job.job_id} for {name}")
    time.sleep(10)

### 2) Download results

In [ ]:
# list results

job_create_date = datetime.strptime('2026-10-02T00:00:00Z', "%Y-%m-%dT%H:%M:%SZ")
job_list = [connection.job(j['id']) for j in connection.list_jobs() if datetime.strptime(j['created'], "%Y-%m-%dT%H:%M:%SZ") > job_create_date]
job_list = job_list[13:]
print(len(job_list), "jobs created after", job_create_date)
job_list

In [ ]:
for i, j in enumerate(job_list):

    print(i, j.describe()["title"], j.status())

    if j.status() == "finished":

        title = j.describe()["title"]
        filepath = os.path.join(results_folder, f"{title}.tiff")

        print(f"Downloading: {filepath}")

        j.get_results().download_file(filepath)

        print("Done!")